# 01. 데이터 살펴보기

두 센서가 **같은 칸·같은 기간**에 화재를 얼마나 다르게 세는지 확인합니다.
여기서 나온 그래프가 "왜 조화가 필요한가"를 보여 주는 발표 자료가 됩니다.

- 입력: `data/processed/` 의 격자 표 (없으면 `python scripts/fetch_sample.py` 로 7일 샘플 생성)
- 7일 샘플은 계절 패턴을 볼 수 없으니, 코드가 도는지 확인하는 용도로만 쓰세요.
- 발표용 그림은 `exports/figures/*.png` 로 저장됩니다 (git 제외, 팀 공유 드라이브에 올리기).
  파일 이름 앞에 격자 표 이름이 붙어서, 샘플로 만든 그림과 실제 데이터 그림이 섞이지 않습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsml import evaluate  # 한글 폰트 설정 포함
from fsml.aggregate import overlap
from fsml.paths import EXPORTS, default_grid

GRID = default_grid()
df = pd.read_parquet(GRID)
print(GRID.name, f"{len(df):,} rows")

# 색: 노트북 08과 같은 계열 (센서 = 파랑·주황, 달력 단계 = 낮음 파랑 · 보통 회색 · 높음 빨강)
SENSOR_COLORS = {"viirs": "#2a78d6", "modis": "#eb6834", "aqua": "#f2a07a", "terra": "#b9a89a"}
LEVEL_COLORS = ["#184f95", "#86b6ef", "#dcdbd6", "#f19b9a", "#b8302f"]

FIGS = EXPORTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)


def save(fig, name: str) -> None:
    path = FIGS / f"{GRID.stem}__{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("saved", path.relative_to(EXPORTS.parent))


df.head()

In [ ]:
period = pd.to_datetime(df["period"])
for sensor in ("modis", "viirs"):
    p = period[df[f"{sensor}_n"].notna()]
    print(f"{sensor.upper()} 운영 기간: {p.min().date()} ~ {p.max().date()}  ({df[f'{sensor}_n'].notna().sum():,} 칸-기간)")
both = overlap(df)
print("학습 가능 (두 센서 모두 운영):", f"{len(both):,}")
if "modis_n_terra" in df:
    ok = df["modis_n"].notna()
    print("Terra + Aqua == modis_n:", bool(((df["modis_n_terra"].fillna(0) + df["modis_n_aqua"].fillna(0))[ok] == df["modis_n"][ok]).all()))
both[["modis_n", "viirs_n", "modis_frp_sum", "viirs_frp_sum"]].describe()

## 1. 같은 칸에서 VIIRS vs MODIS 탐지 수

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
hb = ax.hexbin(np.log1p(both["viirs_n"]), np.log1p(both["modis_n"]),
               gridsize=60, bins="log", cmap="inferno", mincnt=1)
lim = [0, np.log1p(both[["viirs_n", "modis_n"]].to_numpy().max())]
ax.plot(lim, lim, color="0.5", lw=1, ls="--", label="같은 수 (1:1)")
ax.set_xlabel("log1p(VIIRS 탐지 수)")
ax.set_ylabel("log1p(MODIS 탐지 수)")
ax.set_title("같은 칸·같은 기간의 두 센서 탐지 수")
ax.legend(loc="upper left")
fig.colorbar(hb, label="칸 수")
save(fig, "1_scatter_viirs_vs_modis")

## 2. MODIS ÷ VIIRS 비율이 위도대(또는 토지 유형)마다 다른가

비율이 일정하면 VIIRS에 상수 하나만 곱하면 됩니다. 지역마다 다르면 그만큼 환산 모델이 필요합니다.
`landcover_group` 열이 붙어 있으면(`scripts/build_landcover.py`) 토지 유형별 비율도 함께 그립니다.

In [ ]:
band = (np.floor(both["lat"] / 10) * 10).astype(int).rename("위도대")
groups = {"위도대": band}
if "landcover_group" in both:
    groups["토지 유형"] = both["landcover_group"].astype("string").fillna("(없음)").rename("토지 유형")

fig, axes = plt.subplots(1, len(groups), figsize=(6 * len(groups), 5), squeeze=False)
tables = {}
for ax, (name, key) in zip(axes[0], groups.items()):
    t = both.groupby(key)[["modis_n", "viirs_n"]].sum()
    t["modis/viirs"] = t["modis_n"] / t["viirs_n"]
    overall = both["modis_n"].sum() / both["viirs_n"].sum()
    t["modis/viirs"].plot.barh(ax=ax, color=SENSOR_COLORS["modis"])
    ax.axvline(overall, color="0.3", ls="--", lw=1, label=f"전체 {overall:.2f}")
    ax.set_xlabel("MODIS ÷ VIIRS (탐지 수 합 기준)")
    ax.set_ylabel(name + (" 시작 (°)" if name == "위도대" else ""))
    ax.legend()
    tables[name] = t
fig.tight_layout()
save(fig, "2_ratio_by_" + ("zone_landcover" if len(groups) > 1 else "zone"))
pd.concat(tables, names=["기준", "그룹"])

## 3. 연도별 전 세계 합계: 2012년 전후로 두 센서가 어떻게 다른가

MODIS는 2000년부터, VIIRS는 2012년부터 있습니다. 두 선의 높이 차이가 "센서만 바꿔도 화재가 몇 배로 늘어 보이는" 문제입니다.
- 연도 전체를 다 관측하지 못한 해(MODIS 2000년, 마지막 해 등)는 합계가 작게 나오니 점선 표시로 구분합니다.
- 7일 샘플이면 연도가 하나뿐이라 막대 하나만 나옵니다.

In [ ]:
year = pd.to_datetime(df["period"]).dt.year.rename("year")
cols = ["modis_n", "viirs_n"] + (["modis_n_aqua", "modis_n_terra"] if "modis_n_aqua" in df else [])
yearly = df.groupby(year)[cols].sum(min_count=1)
# 그 해에 센서가 운영된 달 수 (12개월 미만이면 불완전한 해)
covered = {s: df[df[f"{s}_n"].notna()].groupby(year)["period"].apply(
    lambda p: pd.to_datetime(p).dt.to_period("M").nunique()) for s in ("modis", "viirs")}

labels = {"modis_n": "MODIS (Terra+Aqua)", "viirs_n": "VIIRS S-NPP",
          "modis_n_aqua": "MODIS Aqua만", "modis_n_terra": "MODIS Terra만"}
styles = {"modis_n": SENSOR_COLORS["modis"], "viirs_n": SENSOR_COLORS["viirs"],
          "modis_n_aqua": SENSOR_COLORS["aqua"], "modis_n_terra": SENSOR_COLORS["terra"]}
fig, ax = plt.subplots(figsize=(10, 4))
if len(yearly) == 1:
    yearly.iloc[0].rename(index=labels).plot.bar(ax=ax, color=[styles[c] for c in cols])
    ax.set_title(f"{yearly.index[0]}년 전 세계 탐지 수 합계 (샘플)")
else:
    for c in cols:
        s = yearly[c]
        full = covered["viirs" if c == "viirs_n" else "modis"].reindex(s.index).fillna(0) >= 12
        ax.plot(s.index, s.where(full), color=styles[c], lw=2 if c in ("modis_n", "viirs_n") else 1,
                marker="o", ms=3, label=labels[c])
        ax.plot(s.index, s, color=styles[c], lw=1, ls=":", alpha=0.6)
    ax.axvline(2012, color="0.4", ls="--", lw=1)
    ax.text(2012.2, ax.get_ylim()[1] * 0.95, "VIIRS 시작", va="top", color="0.3")
    ax.set_title("연도별 전 세계 탐지 수 합계 (점선 = 일부 달만 관측한 해)")
    ax.legend()
ax.set_ylabel("탐지 수")
save(fig, "3_yearly_global_totals")
yearly.assign(**{f"{s}_months": covered[s] for s in covered})

## 4. Terra와 Aqua 중 VIIRS와 짝이 맞는 쪽은?

VIIRS S-NPP와 Aqua는 오후 1:30, Terra는 오전 10:30에 관측합니다.
같은 칸에서 Aqua ÷ VIIRS 비율이 Terra보다 지역마다 덜 흔들리면 "Aqua 기준으로 맞추기"가 더 깔끔하다는 근거가 됩니다.

In [ ]:
MIN_BAND_SHARE = 0.01  # VIIRS 탐지가 전체의 1% 미만인 위도대는 변동계수에서 뺌 (탐지 몇 개짜리가 비율을 왜곡)

if "modis_n_aqua" in both:
    t = both.groupby(band)[["modis_n_terra", "modis_n_aqua", "viirs_n"]].sum()
    r = pd.DataFrame({"Terra ÷ VIIRS": t["modis_n_terra"] / t["viirs_n"],
                      "Aqua ÷ VIIRS": t["modis_n_aqua"] / t["viirs_n"]})
    share = t["viirs_n"] / t["viirs_n"].sum()
    keep = share >= MIN_BAND_SHARE
    ax = r.plot.barh(figsize=(6, 5), color=[SENSOR_COLORS["terra"], SENSOR_COLORS["aqua"]])
    for y, k in enumerate(keep):
        if not k:
            ax.get_yticklabels()[y].set_color("#a3a29e")
    ax.set_xlabel("비율 (탐지 수 합 기준)")
    ax.set_ylabel("위도대 시작 (°) — 회색 글자는 탐지가 적어 변동계수에서 뺌")
    kept = r[keep]
    print(f"변동계수 계산에 쓴 위도대: {list(kept.index)} (VIIRS 탐지 {share[keep].sum():.1%})")
    print("뺀 위도대:", {b: f"{s:.2%}" for b, s in share[~keep].items()})
    print("위도대 간 변동계수 (작을수록 일정):", (kept.std() / kept.mean()).round(3).to_dict())
    display(r.assign(VIIRS_탐지_비중=share).round(3))
else:
    print("격자 표에 modis_n_terra/aqua 열이 없습니다. scripts/aggregate.py로 다시 만드세요.")

## 5. 왜 조화가 필요한가: 같은 지역의 화재 활동 달력

노트북 08과 같은 **화재 활동 지수**(같은 달 평년 대비 백분위, 5단계)로 그린 달력입니다. 평년은 VIIRS가 없던 2003~2011년의 MODIS입니다.
- 왼쪽: 2012년부터 MODIS 대신 **VIIRS 탐지 수를 그대로** 이어 붙인 경우.
- 오른쪽: 처음부터 끝까지 MODIS만 쓴 경우 (조화된 달력이 맞춰야 할 모습).

VIIRS가 더 많이 잡는 효과는 불이 많은 **화재철 달**에 몰려서 나타납니다. 불이 거의 없는 달은 두 센서 모두 0 근처라 차이가 작습니다.
그래서 2012년 이후 전체 달이 아니라 **화재철 달이 거의 다 "매우 높음"으로 바뀌는 것**이 핵심입니다
(실제 데이터, 아마존 남부: "매우 높음" 비율이 전체 달 기준 6% → 29%, 화재철 8~10월은 거의 전부).
아래 표에 지역별로 전체 달과 화재철 달의 비율을 함께 냅니다.

지역은 두 곳을 그립니다. 발표에는 차이가 더 분명한 쪽을 쓰세요.
- 아마존 남부: 08 노트북의 달력 예시와 같은 지역. 해마다 불 양이 크게 달라짐
- 아프리카 남부 사바나(앙골라 북부·콩고민주공화국 남부): 해마다 불 양이 비교적 일정해서, 센서 차이만 따로 보기 좋음

연도가 5년 미만이면(7일 샘플 등) 평년을 만들 수 없어 그리지 않습니다.

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap

from fsml.activity import LEVEL_EDGES, LEVELS, MIN_BASELINE_YEARS, monthly_totals, percentile

AOIS = {
    "amazon": {"name": "아마존 남부 (브라질 마투그로수·파라)", "lat": (-15, -5), "lon": (-60, -50)},  # 08과 같은 지역
    "savanna": {"name": "아프리카 남부 사바나 (앙골라·콩고민주공화국)", "lat": (-12, -2), "lon": (15, 25)},
}
NAIVE_BASE = (2003, 2011)
VIIRS_FROM = pd.Timestamp("2012-02-01")  # 2012-01은 VIIRS가 1월 20일부터라 뺌
FIRE_SEASON_MONTHS = 3  # 평년 MODIS 합계가 가장 큰 3개월 = 화재철
cmap, norm = ListedColormap(LEVEL_COLORS), BoundaryNorm([0, *LEVEL_EDGES, 100.01], len(LEVELS))


def naive_vs_modis(aoi: dict):
    box = df[df["lat"].between(*aoi["lat"]) & df["lon"].between(*aoi["lon"])].assign(region=aoi["name"])
    t = monthly_totals(box, ["modis_n", "viirs_n"], "region")
    t["naive"] = np.where(t["period"] >= VIIRS_FROM, t["viirs_n"], t["modis_n"])
    t.loc[t["period"].between(pd.Timestamp("2012-01-01"), VIIRS_FROM, inclusive="left"), "naive"] = np.nan
    pct = {"VIIRS를 그대로 이어 붙임": percentile(t, "naive", "region", base_col="modis_n", years=NAIVE_BASE),
           "MODIS만": percentile(t, "modis_n", "region", years=NAIVE_BASE)}
    month = t["period"].dt.month
    base = t[t["period"].dt.year.between(*NAIVE_BASE)]
    season = base.groupby(base["period"].dt.month)["modis_n"].sum().nlargest(FIRE_SEASON_MONTHS).index
    return t, pct, month.isin(season).to_numpy(), sorted(season)


years_in_grid = pd.to_datetime(df["period"]).dt.year
if years_in_grid.nunique() < MIN_BASELINE_YEARS or years_in_grid.min() > NAIVE_BASE[0]:
    print(f"연도가 모자라 달력을 그리지 않습니다 ({years_in_grid.nunique()}개 연도). 실제 월 격자에서 실행하세요.")
else:
    summary = {}
    for key, aoi in AOIS.items():
        t, pct, in_season, season = naive_vs_modis(aoi)
        fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True)
        for ax, (title, p) in zip(axes, pct.items()):
            cal = t.assign(pct=p.to_numpy(), year=t["period"].dt.year, month=t["period"].dt.month) \
                   .pivot(index="year", columns="month", values="pct")
            mesh = ax.pcolormesh(cal.columns, cal.index, cal.to_numpy(), cmap=cmap, norm=norm,
                                 edgecolors="#fcfcfb", linewidth=1.5)
            ax.axhline(2011.5, color="#52514e", lw=1.5, ls="--")
            ax.set_xticks(range(1, 13), [f"{m}월" + ("*" if m in season else "") for m in range(1, 13)])
            ax.set_yticks(cal.index)
            ax.set_title(title)
        axes[0].invert_yaxis()
        cbar = fig.colorbar(mesh, ax=axes, ticks=[5, 20, 50, 80, 95], fraction=0.03, pad=0.02)
        cbar.ax.set_yticklabels(LEVELS)
        fig.suptitle(f"화재 활동 지수 (평년 {NAIVE_BASE[0]}~{NAIVE_BASE[1]} MODIS 대비): {aoi['name']}\n"
                     f"* 화재철 (평년 MODIS 합계 상위 {FIRE_SEASON_MONTHS}개월) · 점선 = VIIRS 시작 (2012)")
        save(fig, f"5_calendar_naive_vs_modis_{key}")

        after = (t["period"] >= VIIRS_FROM).to_numpy()
        for title, p in pct.items():
            high = p.to_numpy() >= 90
            summary[(aoi["name"], title)] = {
                "화재철": "·".join(f"{m}월" for m in season),
                "매우 높음: 2012 이후 전체 달": high[after].mean(),
                "매우 높음: 2012 이후 화재철 달": high[after & in_season].mean(),
                "매우 높음: 평년(2003~2011) 화재철 달": high[~after & in_season & t["period"].dt.year.ge(NAIVE_BASE[0]).to_numpy()].mean(),
            }
    display(pd.DataFrame(summary).T.style.format({c: "{:.0%}" for c in list(summary.values())[0] if c != "화재철"}))

## 메모 (TODO)

- 비율이 위도대·계절·토지 유형에 따라 얼마나 달라지는지 정리 (2번 그림, 토지 유형은 `landcover_group`이 붙은 뒤)
- VIIRS는 0인데 MODIS만 잡은 칸, 또는 그 반대가 얼마나 되는지 확인
- 화산·공장 같은 정적 열원을 포함했을 때(`aggregate.py --all-types`)와 비교
- Terra/Aqua 비교는 노트북 07에서 결론(MODIS 전체 유지)